# Solute and water transport parameters

In [1]:
from openalea.widgets.plantgl import PlantGL # notebook viewer 3D
from openalea.plantgl.algo.view import view # 2D view

from openalea.hydroroot.main import root_builder, hydroroot_solute_flow
from openalea.hydroroot.init_parameter import Parameters
from openalea.hydroroot.read_file import read_archi_data
from openalea.hydroroot.display import mtg_scene


Read the yaml file and set the Parameters variables, assuming that the code is run from the example folder

In [2]:
parameter = Parameters()
parameter.read_file('parameters_Ctr-3P2.yml')

Read the architecture file and build the MTG

In [3]:
fname = parameter.archi['input_dir'] + parameter.archi['input_file'][0]
df = read_archi_data(fname)
g, primary_length, total_length, surface, seed = root_builder( primary_length = parameter.archi['primary_length'],
                                                                delta = parameter.archi['branching_delay'],
                                                                nude_length = parameter.archi['nude_length'], 
                                                                df = df,
                                                                segment_length = parameter.archi['segment_length'],
                                                                length_data = parameter.archi['length_data'],
                                                                order_max = parameter.archi['order_max'],
                                                                order_decrease_factor = parameter.archi['order_decrease_factor'],
                                                                ref_radius = parameter.archi['ref_radius'])

In the code the concentration are in $mol.\mu L^{-1}$, then we convert them from $mol.m^{-3}$

In [4]:
Cse = parameter.solute['Cse'] * 1e-9 # mol/m3 -> mol/microL, external permeating solute concentration
Ce = parameter.solute['Ce'] * 1e-9 # mol/m3 -> mol/microL, external non-permeating solute concentration

Perform the calculation

In [5]:
g, Jv = hydroroot_solute_flow(g, psi_e = parameter.exp['psi_e'],
                                psi_base = parameter.exp['psi_base'],
                                k0 = parameter.hydro['k0'],
                                axial_conductivity_data = parameter.hydro['axial_conductance_data'],
                                J_s = parameter.solute['J_s'],
                                Ps=parameter.solute['P_s'],
                                Cse = Cse,
                                Ce=Ce,
                                sigma = parameter.solute['Sigma'])

In [6]:
result=f"""
primary length (m): {primary_length}
surface (m2): {surface}
total length (m): {total_length}
flux (microL/s): {Jv}
"""
print(result)


primary length (m): 0.434
surface (m2): 0.005643500494241343
total length (m): 3.979
flux (microL/s): 0.025700314390474904



Display the concentration in the architecture 3D view

In [7]:
s = mtg_scene(g, prop_cmap = 'C') # create a scene from the mtg with the property j is the radial flux in ul/s
PlantGL(s)

Plot(antialias=3, axes=['x', 'y', 'z'], axes_helper=1.0, axes_helper_colors=[16711680, 65280, 255], background…

In [8]:
g, Jv = hydroroot_solute_flow(g, psi_e = parameter.exp['psi_base'],
                                psi_base = parameter.exp['psi_base'],
                                k0 = parameter.hydro['k0'],
                                axial_conductivity_data = parameter.hydro['axial_conductance_data'],
                                J_s = parameter.solute['J_s'],
                                Ps=parameter.solute['P_s'],
                                Cse = Cse,
                                Ce=Ce,
                                sigma = parameter.solute['Sigma'])
result=f"""
primary length (m): {primary_length:.3f}
surface (m2): {surface:.3e}
total length (m): {total_length:.3f}
flux (microL/s): {Jv:.4f}
"""
print(result)


primary length (m): 0.434
surface (m2): 5.644e-03
total length (m): 3.979
flux (microL/s): 0.0072



#### Adjustement of Js

Let try to adjust the active pumping rate `Js` to simulate a higher flux `Jv_to_fit`

The adjustement is done by a simple Newton-Raphson scheme

In [9]:
Jv_to_fit = 0.01
Js_old = parameter.solute['J_s']
F_old = (Jv - Jv_to_fit) ** 2.0  # the objective function
Js = 1.1*parameter.solute['J_s']  # to initiate a simulation in the loop to compare with the previous one
eps = 1e-10  # the accuracy wanted
F = 1.  # to launch the loop
# Newton-Raphson loop to get Js
while (F > eps):
    g, Jv = hydroroot_solute_flow(g, psi_e = parameter.exp['psi_e'],
                                    psi_base = parameter.exp['psi_base'],
                                    k0 = parameter.hydro['k0'],
                                    axial_conductivity_data = parameter.hydro['axial_conductance_data'],
                                    J_s = Js,
                                    Ps=parameter.solute['P_s'],
                                    Cse = Cse,
                                    Ce=Ce,
                                    sigma = parameter.solute['Sigma'])

    F = (Jv - Jv_to_fit) ** 2.0  # the objective function
    if abs(F) > eps:
        dfdJs = (F - F_old) / (Js - Js_old)  # the derivative of F according to Js
        Js_old = Js
        Js = Js_old - F / dfdJs  # new estimate
        F_old = F

In [10]:
result=f"""
Jv to fit: {Jv_to_fit:.4f}
Simulated Jv: {Jv:.4f}
Js 1st guess: {parameter.solute['J_s']:.3e}
adjusted Js: {Js:.3e}
"""
print(result)


Jv to fit: 0.0100
Simulated Jv: 0.0100
Js 1st guess: 1.714e-07
adjusted Js: 2.864e-07

